# 실험 제목
- 담당: 김영빈
- 날짜: 26/09/21
- 목적: json 데이터를 csv파일 하나로 합치기

> 끝나면 결과를 `experiments/LOG.md`에 한 줄 남기기

## 1단계. 라이브러리 및 경로 설정

JSON을 읽고 표 형태로 변환하기 위해 `json`, `pathlib`, `pandas`를 사용한다. 이후 전체 파일을 처리할 때 진행률을 확인할 수 있도록 `tqdm`도 불러온다.

노트북을 저장소 루트 또는 `playground/youngbeen`에서 실행해도 같은 데이터 경로를 사용하도록 현재 위치에서 프로젝트 루트를 탐색한다. 입력 JSON과 생성할 CSV는 모두 Git 추적에서 제외된 `data/processed` 아래에 둔다. 현재 데이터는 training 데이터이므로 split 컬럼은 생성하지 않는다.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm


def find_project_root(start_path: Path) -> Path:
    """현재 위치부터 상위 폴더를 확인해 프로젝트 루트를 찾는다."""
    start_path = start_path.resolve()

    for candidate in (start_path, *start_path.parents):
        if (candidate / ".git").exists() and (candidate / "data" / "processed").exists():
            return candidate

    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. dontalk 저장소 내부에서 노트북을 실행해 주세요."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_PATH = PROCESSED_DIR / "qa_flat_train.csv"

if not PROCESSED_DIR.is_dir():
    raise NotADirectoryError(f"데이터 디렉토리가 없습니다: {PROCESSED_DIR}")

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"입력 디렉토리: {PROCESSED_DIR}")
print(f"출력 CSV: {OUTPUT_PATH}")
print(f"pandas 버전: {pd.__version__}")

## 2단계. 전체 JSON 파일 탐색 및 개수 확인

`data/processed` 아래의 모든 하위 디렉토리를 재귀적으로 탐색해 `.json` 파일 목록을 만든다. 은행·보험·증권의 폴더 번호를 코드에 직접 나열하지 않으므로, 새로운 하위 폴더가 추가되어도 자동으로 포함된다.

파일 목록은 항상 같은 순서로 처리할 수 있도록 정렬한다. 이 단계에서는 JSON 내부 내용은 아직 읽지 않고, 발견한 파일 수와 최상위 데이터 폴더별 개수만 확인한다.

In [ ]:
from collections import Counter


json_files = sorted(
    path for path in PROCESSED_DIR.rglob("*.json")
    if path.is_file()
)

if not json_files:
    raise FileNotFoundError(f"JSON 파일을 찾지 못했습니다: {PROCESSED_DIR}")

folder_counts = Counter(
    path.relative_to(PROCESSED_DIR).parts[0]
    for path in json_files
)

for folder_name, count in sorted(folder_counts.items()):
    print(f"[{folder_name}] {count:,}개")

print(f"\n총 JSON 파일 개수: {len(json_files):,}개")
print(f"첫 번째 파일: {json_files[0].relative_to(PROCESSED_DIR)}")
print(f"마지막 파일: {json_files[-1].relative_to(PROCESSED_DIR)}")

## 3단계. 파일명 메타데이터 추출

각 JSON 파일명은 `데이터분야_도메인_주제_원천일련번호_QA일련번호.json` 형식이다. 파일명을 `_` 기준으로 나눠 도메인과 상담 주제를 사람이 읽기 쉬운 값으로 변환한다.

파일명 형식이나 코드가 예상과 다른 파일은 조용히 제외하지 않고 오류 목록에 모아 즉시 확인한다. 이 단계에서는 JSON 내부 내용은 아직 읽지 않는다.

In [ ]:
DOMAIN_MAP = {
    "bk": "은행",
    "ins": "보험",
    "sec": "증권",
}

CATEGORY_MAP = {
    "bk": {
        "01": "거래내역/잔액조회",
        "02": "중계요청/착오송금",
        "03": "자동이체조회",
        "04": "만기,연장/해지,수신",
        "05": "금융거래한도/비대면한도계좌",
        "06": "이자/연체금액",
        "07": "부수거래금리감면",
        "08": "대출문의(만기/연장/조회등)",
        "09": "환전문의",
    },
    "ins": {
        "01": "자동차보험상담",
        "02": "자동차사고접수",
        "03": "계약내용변경/해지",
        "04": "기타계약관련문의",
        "05": "보험금청구",
    },
    "sec": {
        "01": "HTS/MTS",
        "02": "계좌관리",
        "03": "신용거래/담보대출",
        "04": "자금이체/계좌제한",
        "05": "절세형금융상품",
        "06": "주식주문",
        "07": "증권계좌조회",
        "08": "해외주문",
    },
}


def parse_filename_metadata(json_path: Path) -> dict:
    """JSON 파일명에서 데이터셋 메타데이터를 추출한다."""
    parts = json_path.stem.split("_")
    if len(parts) != 5:
        raise ValueError(f"예상한 파일명 형식이 아닙니다: {json_path.name}")

    data_field_code, domain_code, topic_code, source_sequence, qa_sequence = parts

    if domain_code not in DOMAIN_MAP:
        raise ValueError(f"알 수 없는 도메인 코드입니다: {json_path.name}")
    if topic_code not in CATEGORY_MAP[domain_code]:
        raise ValueError(f"알 수 없는 주제 코드입니다: {json_path.name}")

    return {
        "file_name": json_path.name,
        "relative_path": str(json_path.relative_to(PROCESSED_DIR)),
        "data_field_code": data_field_code,
        "domain_code": domain_code,
        "domain_name": DOMAIN_MAP[domain_code],
        "topic_code": topic_code,
        "mapped_topic": CATEGORY_MAP[domain_code][topic_code],
        "source_sequence": source_sequence,
        "qa_sequence": qa_sequence,
    }


file_metadata = []
filename_errors = []

for json_path in tqdm(json_files, desc="파일명 분석"):
    try:
        file_metadata.append(parse_filename_metadata(json_path))
    except ValueError as error:
        filename_errors.append({"path": str(json_path), "error": str(error)})

if filename_errors:
    error_preview = pd.DataFrame(filename_errors).head()
    raise ValueError(
        f"파일명 분석 실패: {len(filename_errors):,}개\n{error_preview.to_string(index=False)}"
    )

metadata_df = pd.DataFrame(file_metadata)

print(f"파일명 분석 성공: {len(metadata_df):,}개")
print("\n도메인별 파일 개수:")
print(metadata_df["domain_name"].value_counts().to_string())

metadata_df.head()

## 4단계. JSON 본문 펼치기 및 하나의 DataFrame으로 병합

각 파일의 JSON 본문을 읽어 `source`, `consulting`, `qa_data` 영역을 한 행으로 펼친다. 같은 이름의 키가 충돌하지 않도록 원래 영역명을 컬럼 접두사로 붙인다. 예를 들어 `source.source_id`는 `source_source_id`, `qa_data[0].input.question`은 `qa_data_input_question`이 된다.

3단계에서 만든 파일명 메타데이터와 펼친 JSON 값을 합쳐 `df`를 만든다. 현재 데이터 규칙에 따라 JSON 하나당 `qa_data` 항목이 정확히 하나인지 검사하며, 읽기·구조 오류가 있는 파일은 누락시키지 않고 오류로 중단한다. 이 단계에서는 아직 CSV 파일을 저장하지 않는다.

In [ ]:
def flatten_json_record(data: dict) -> dict:
    """중첩된 JSON 한 건을 CSV의 한 행으로 사용할 수 있게 펼친다."""
    result = {}

    for section_name in ("source", "consulting"):
        section = data.get(section_name)
        if not isinstance(section, dict):
            raise ValueError(f"{section_name}가 dictionary 형식이 아닙니다.")

        for key, value in section.items():
            result[f"{section_name}_{key}"] = value

    qa_items = data.get("qa_data")
    if not isinstance(qa_items, list):
        raise ValueError("qa_data가 list 형식이 아닙니다.")
    if len(qa_items) != 1:
        raise ValueError(f"qa_data 항목 수가 1개가 아닙니다: {len(qa_items)}개")

    qa = qa_items[0]
    if not isinstance(qa, dict):
        raise ValueError("qa_data[0]이 dictionary 형식이 아닙니다.")

    for key, value in qa.items():
        if key == "input":
            if not isinstance(value, dict):
                raise ValueError("qa_data[0].input이 dictionary 형식이 아닙니다.")

            for input_key, input_value in value.items():
                result[f"qa_data_input_{input_key}"] = input_value
        else:
            result[f"qa_data_{key}"] = value

    return result


merged_rows = []
json_errors = []

for metadata in tqdm(file_metadata, desc="JSON 병합"):
    json_path = PROCESSED_DIR / metadata["relative_path"]

    try:
        with json_path.open("r", encoding="utf-8") as file:
            json_data = json.load(file)

        flattened_data = flatten_json_record(json_data)
        merged_rows.append({**metadata, **flattened_data})
    except (OSError, UnicodeDecodeError, json.JSONDecodeError, TypeError, ValueError) as error:
        json_errors.append({
            "relative_path": metadata["relative_path"],
            "error": str(error),
        })

if json_errors:
    error_preview = pd.DataFrame(json_errors).head(10)
    raise ValueError(
        f"JSON 읽기 또는 구조 검사 실패: {len(json_errors):,}개\n"
        f"{error_preview.to_string(index=False)}"
    )

df = pd.DataFrame(merged_rows)

required_columns = [
    "source_source_id",
    "consulting_consulting_category",
    "consulting_consulting_topic",
    "qa_data_qa_id",
    "qa_data_instruction",
    "qa_data_input_question",
    "qa_data_input_answer",
    "qa_data_output",
]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise KeyError(f"필수 컬럼이 없습니다: {missing_columns}")

if len(df) != len(json_files):
    raise ValueError(
        f"파일 수와 DataFrame 행 수가 다릅니다: {len(json_files):,}개 / {len(df):,}행"
    )

missing_required_values = df[required_columns].isna().sum()

print(f"병합 완료: {len(df):,}행 × {len(df.columns):,}열")
print(f"처리 오류: {len(json_errors):,}개")
print("\n필수 컬럼별 결측값 개수:")
print(missing_required_values.to_string())

df.head()

## 5단계. 데이터 검증 및 CSV 저장

저장 전에 전체 행 수, 파일명과 QA ID의 중복, 필수 텍스트의 빈 값, 파일명과 JSON 내부 QA ID의 일치 여부를 검사한다. 파일명에서 만든 도메인·주제 값이 JSON 내부의 상담 분류와 같은지도 확인한다.

행 수·중복·필수값·QA ID 검사는 저장을 중단하는 필수 검사로 사용한다. 파일명 매핑과 JSON 내부 분류가 다른 경우에는 원본 값을 임의 수정하거나 제외하지 않고 문제 건수를 보고한 뒤 그대로 보존한다. 한글이 Excel에서도 깨지지 않도록 `utf-8-sig` 인코딩을 사용하며, DataFrame 인덱스와 데이터 분할 컬럼은 저장하지 않는다.

In [ ]:
expected_row_count = len(json_files)
duplicate_file_count = int(df["file_name"].duplicated().sum())
duplicate_qa_id_count = int(df["qa_data_qa_id"].duplicated().sum())

required_text_columns = [
    "qa_data_instruction",
    "qa_data_input_question",
    "qa_data_input_answer",
    "qa_data_output",
]
blank_required_counts = {
    column: int(df[column].fillna("").astype(str).str.strip().eq("").sum())
    for column in required_text_columns
}

filename_ids = df["file_name"].map(lambda name: Path(name).stem)
qa_id_mismatch_count = int(filename_ids.ne(df["qa_data_qa_id"]).sum())
domain_mismatch_count = int(
    df["domain_name"].ne(df["consulting_consulting_category"]).sum()
)
topic_mismatch_count = int(
    df["mapped_topic"].ne(df["consulting_consulting_topic"]).sum()
)

validation_results = {
    "행 수 불일치": abs(expected_row_count - len(df)),
    "중복 파일명": duplicate_file_count,
    "중복 QA ID": duplicate_qa_id_count,
    "파일명-QA ID 불일치": qa_id_mismatch_count,
    "도메인 불일치": domain_mismatch_count,
    "주제 불일치": topic_mismatch_count,
    **{f"빈 값: {column}": count for column, count in blank_required_counts.items()},
}
validation_df = pd.DataFrame(
    validation_results.items(),
    columns=["검증 항목", "문제 건수"],
)

blocking_check_names = [
    "행 수 불일치",
    "중복 파일명",
    "중복 QA ID",
    "파일명-QA ID 불일치",
    *[f"빈 값: {column}" for column in required_text_columns],
]
failed_checks = validation_df.loc[
    validation_df["검증 항목"].isin(blocking_check_names)
    & validation_df["문제 건수"].gt(0)
]
if not failed_checks.empty:
    raise ValueError(
        "저장 전 데이터 검증에 실패했습니다.\n"
        f"{failed_checks.to_string(index=False)}"
    )

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("저장 전 필수 검증 통과")
print(
    f"원본 분류 불일치: 도메인 {domain_mismatch_count:,}개, "
    f"주제 {topic_mismatch_count:,}개 (수정·제외 없이 보존)"
)
print(f"CSV 저장 완료: {OUTPUT_PATH}")
print(f"저장 데이터: {len(df):,}행 × {len(df.columns):,}열")

validation_df

## 6단계. 저장된 CSV 재검증

방금 저장한 CSV를 문자열 형식으로 다시 읽어 원본 DataFrame과 비교한다. 행·열 수, 컬럼 이름과 순서, QA ID 순서, 도메인별 행 수가 모두 같아야 완료로 판단한다.

이 검사는 CSV가 실제로 정상 저장되었는지 확인하는 단계다. 데이터 분할이나 새로운 컬럼 생성은 수행하지 않는다.

In [ ]:
if not OUTPUT_PATH.is_file():
    raise FileNotFoundError(f"저장된 CSV 파일이 없습니다: {OUTPUT_PATH}")

saved_df = pd.read_csv(
    OUTPUT_PATH,
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
)

if saved_df.shape != df.shape:
    raise ValueError(f"저장 전후 크기가 다릅니다: {df.shape} / {saved_df.shape}")

if saved_df.columns.tolist() != df.columns.tolist():
    raise ValueError("저장 전후 컬럼 이름 또는 순서가 다릅니다.")

expected_qa_ids = df["qa_data_qa_id"].astype(str).reset_index(drop=True)
saved_qa_ids = saved_df["qa_data_qa_id"].reset_index(drop=True)
if not saved_qa_ids.equals(expected_qa_ids):
    raise ValueError("저장 전후 QA ID 또는 행 순서가 다릅니다.")

expected_domain_counts = (
    df["domain_name"].astype(str).value_counts().sort_index()
)
saved_domain_counts = saved_df["domain_name"].value_counts().sort_index()
if not saved_domain_counts.equals(expected_domain_counts):
    raise ValueError("저장 전후 도메인별 행 수가 다릅니다.")

csv_size_mb = OUTPUT_PATH.stat().st_size / (1024 ** 2)

print("CSV 재검증 통과")
print(f"파일 경로: {OUTPUT_PATH}")
print(f"파일 크기: {csv_size_mb:,.1f} MB")
print(f"행과 열: {saved_df.shape[0]:,}행 × {saved_df.shape[1]:,}열")
print("분할 컬럼: 생성하지 않음")
print("\n도메인별 행 수:")
print(saved_domain_counts.to_string())

saved_df.head()